# Factory IO 모드버스 연결제어

### modbus_water01 기준으로 연결
### 데이터센터 냉각수제어

In [1]:
# 모드버스접속
from pymodbus.client import ModbusTcpClient
import time as tt
import threading

client = ModbusTcpClient('210.119.14.58', port=502)
client.connect()

True

### 아날로그 출력제어

In [2]:
# Fill Value 제어
Fvalve = 35
client.write_register(0,Fvalve) # 40001 에 어떠한 값을 보내기

In [3]:
# Discharge Value 제어
Dvalve = 100
client.write_register(1,Dvalve) # 40001 에 어떠한 값을 보내기

In [4]:
# registers 로 여러주소동시제어
# client.write_registers(0, [Fvalve,Dvalve])
# 초기화 시키기
client.write_registers(0, [20,20])

In [5]:
hr= client.read_holding_registers(0,count=4) # 40001 ~ 40004 까지 읽어오기
hr.registers

[20, 20, 0, 0]

# 아날로그 입력 모니터링

In [ ]:
scan_run = True
def meter_scan():
    while scan_run:
        water_Meter = client.read_input_registers(0, count=4).registers # 30001 ~ 30004 까지 읽어오기
        print(f"\r물의 수위: {water_Meter[0]}% / 배수량: {water_Meter[1]}L/s", end="")
        
        if ( 70 - water_Meter[0]) < 1 :
            client.write_register(0,0) # 40001 에 어떠한 값을 보내기
        if ( 70 - water_Meter[0]) < 5 :
            client.write_register(0,5) # 40001 에 어떠한 값을 보내기
        elif (70 - water_Meter[0]) <= 10 :
            client.write_register(0,20)
        elif (70 - water_Meter[0]) <= 20 :
            client.write_register(0,50)
        elif (70 - water_Meter[0]) <= 50 :
            client.write_register(0,80)
        elif (70 - water_Meter[0]) <= 100 :
            client.write_register(0,100)
        tt.sleep(0.1)
scan = threading.Thread(target=meter_scan, daemon=True)
print("모니터링시작")
scan.start()

모니터링시작
물의 수위: 100% / 배수량: 45L/s

물의 수위: 37% / 배수량: 45L/ss

In [ ]:
scan_run = True

물의 수위: 100% / 배수량: 45L/s

물의 수위: 100% / 배수량: 45L/s

In [ ]:
import time as tt
import threading

scan_run = True

# 목표 수위
SETPOINT = 70.0

# PID Gain
Kp = 2.5
Ki = 0.15
Kd = 0.8

integral = 0
previous_error = 0
previous_time = tt.time()

def meter_scan():

    global integral
    global previous_error
    global previous_time

    while scan_run:

        water_Meter = client.read_input_registers(
            0,
            count=4
        ).registers

        level = water_Meter[0]

        print(
            f"\r물의 수위:{level}% / 배수량:{water_Meter[1]}L/s",
            end=""
        )

        now = tt.time()
        dt = now - previous_time

        if dt <= 0:
            dt = 0.1

        # ----------------------
        # PID
        # ----------------------
        error = SETPOINT - level

        # Integral
        integral += error * dt

        # Derivative
        derivative = (error - previous_error) / dt

        output = (
            Kp * error
            + Ki * integral
            + Kd * derivative
        )

        previous_error = error
        previous_time = now

        # 출력 제한
        output = max(0, min(100, output))

        client.write_register(
            0,
            int(output)
        )

        tt.sleep(0.1)

scan = threading.Thread(
    target=meter_scan,
    daemon=True
)

print("모니터링 시작")

scan.start()

모니터링 시작
물의 수위: 42% / 배수량: 45L/s

물의 수위: 0% / 배수량: 0L/ssss